In [2]:
import polars as pl
import xgboost as xgb

In [3]:
train=pl.read_parquet('../data/train_and_validation/test.parquet')
train_labels=pl.read_parquet('../data/train_and_validation/test_labels.parquet')

In [4]:
train.head()

session,aid,ts,type
i32,i32,i32,u8
11098528,11830,1661119200,0
11098529,1105029,1661119200,0
11098530,264500,1661119200,0
11098530,264500,1661119288,0
11098530,409236,1661119369,0


In [24]:
train_labels.head(),train.shape

(shape: (5, 4)
 ┌──────────┬──────┬─────────┬─────┐
 │ session  ┆ type ┆ aid     ┆ gt  │
 │ ---      ┆ ---  ┆ ---     ┆ --- │
 │ i32      ┆ u8   ┆ i32     ┆ i32 │
 ╞══════════╪══════╪═════════╪═════╡
 │ 11098528 ┆ 0    ┆ 1679529 ┆ 1   │
 │ 11098528 ┆ 1    ┆ 1199737 ┆ 1   │
 │ 11098528 ┆ 2    ┆ 990658  ┆ 1   │
 │ 11098528 ┆ 2    ┆ 950341  ┆ 1   │
 │ 11098528 ┆ 2    ┆ 1462506 ┆ 1   │
 └──────────┴──────┴─────────┴─────┘,
 (7683577, 9))

In [6]:
# with_column在不修改元DataFrame的前提下，新增或替换一列，返回新的DataFrame
# over相当group_by，需要先定义算什么，再定义在哪算，因此在统计函数之后调用

# 为session赋予按照时间倒序的编号(从0开始)
def add_action_num_reverse_chrono(df):
    return df.select([
        pl.col('*'),
        pl.col('session').cum_count().over('session').alias('action_num_reverse_chrono'),
    ])

# 计算session内的行为总数
def add_session_length(df):
    return df.select([
        pl.col('*'),
        pl.col('session').count().over('session').alias('session_length'),
    ])

# 求解时间位置权重，时间越近，权重越大，将[1,L]映射到[0.1,1],公式为y=y_min+(y_max-y_min)/(x_max-x_min)*pos
def add_lo_recency_score(df):
    return df.with_columns(
        pl.when(pl.col("session_length") == 1)
          .then(1.0)
          .otherwise(
              2 ** (
                  0.1
                  + (1 - 0.1)
                    / (pl.col("session_length") - 1)
                    * (
                        pl.col("action_num_reverse_chrono")
                        - 1
                    )
              ) - 1
          )
          .alias("log_recency_score")
    )


# 按照操作赋予权重
def add_type_weighted_log_recency_score(df):
    type_weight={0:1,1:6,2:3}
    type_weighted_log_recency_score=pl.Series(df['type'].replace(type_weight)*df['log_recency_score'])
    return df.with_columns(type_weighted_log_recency_score.alias('type_weighted_log_recency_score').alias('type_weighted_log_recency_score'))

def apply(df,pipeline):
    for f in pipeline:
        df=f(df)
    return df

In [7]:
pipeline=[add_action_num_reverse_chrono,add_session_length,add_lo_recency_score,add_type_weighted_log_recency_score]
train=apply(train,pipeline)
type2id={'clicks':0,'carts':1,'orders':2}

In [8]:
train.head()

session,aid,ts,type,action_num_reverse_chrono,session_length,log_recency_score,type_weighted_log_recency_score
i32,i32,i32,u8,u32,u32,f64,f64
11098528,11830,1661119200,0,1,1,1.0,1.0
11098529,1105029,1661119200,0,1,1,1.0,1.0
11098530,264500,1661119200,0,1,6,0.071773,0.071773
11098530,264500,1661119288,0,2,6,0.214195,0.214195
11098530,409236,1661119369,0,3,6,0.375542,0.375542


In [9]:
# explode把列表拆成多行
train_labels=train_labels.explode('ground_truth').with_columns([
    pl.col('ground_truth').alias('aid'),
    pl.col('type').replace(type2id)
])[['session','type','aid']]

train_labels=train_labels.with_columns([
    pl.col('session').cast(pl.datatypes.Int32),
    pl.col('type').cast(pl.datatypes.UInt8),
    pl.col('aid').cast(pl.datatypes.Int32),
])

# lit创建一个常量，gt定义正样本
train_labels=train_labels.with_columns(pl.lit(1).alias('gt'))

train=train.join(train_labels,how='left',on=['session','type','aid']).with_columns(pl.col('gt').fill_null(0))

In [10]:
train.head(),train.columns,train.shape

(shape: (5, 9)
 ┌──────────┬─────────┬────────────┬──────┬───┬───────────────┬───────────────┬───────────────┬─────┐
 │ session  ┆ aid     ┆ ts         ┆ type ┆ … ┆ session_lengt ┆ log_recency_s ┆ type_weighted ┆ gt  │
 │ ---      ┆ ---     ┆ ---        ┆ ---  ┆   ┆ h             ┆ core          ┆ _log_recency_ ┆ --- │
 │ i32      ┆ i32     ┆ i32        ┆ u8   ┆   ┆ ---           ┆ ---           ┆ scor…         ┆ i32 │
 │          ┆         ┆            ┆      ┆   ┆ u32           ┆ f64           ┆ ---           ┆     │
 │          ┆         ┆            ┆      ┆   ┆               ┆               ┆ f64           ┆     │
 ╞══════════╪═════════╪════════════╪══════╪═══╪═══════════════╪═══════════════╪═══════════════╪═════╡
 │ 11098528 ┆ 11830   ┆ 1661119200 ┆ 0    ┆ … ┆ 1             ┆ 1.0           ┆ 1.0           ┆ 0   │
 │ 11098529 ┆ 1105029 ┆ 1661119200 ┆ 0    ┆ … ┆ 1             ┆ 1.0           ┆ 1.0           ┆ 1   │
 │ 11098530 ┆ 264500  ┆ 1661119200 ┆ 0    ┆ … ┆ 6             ┆ 0.0

In [11]:
feature_cols = [
    "aid",
    "type",
    "action_num_reverse_chrono",
    "session_length",
    "log_recency_score",
    "type_weighted_log_recency_score",
]
label_col = "gt"

In [12]:
train=train.sort(['session'])

In [13]:
import numpy as np
X=train.select(feature_cols).to_numpy().astype(np.float32)
y=train.select(label_col).to_numpy().astype(np.int32)

In [14]:
group_sizes = (
    train.group_by("session")
      .len()
      .sort("session")["len"]
      .to_list()
)

group_sizes[:5]

[1, 1, 6, 24, 2]

In [15]:
dtrain = xgb.DMatrix(X, label=y)
dtrain.set_group(group_sizes)

params={
    "objective": "rank:pairwise",
    "eval_metric": "ndcg",

    # GPU
    "tree_method": "gpu_hist",
    "device": "cuda",
}

In [16]:
ranker=xgb.train(params,dtrain,num_boost_round=500)

/home/mingyu/miniconda3/envs/xgboost/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [01:32:43] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)


In [19]:
ranker.save_model('../model/xgboost.model')

/home/mingyu/miniconda3/envs/xgboost/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [01:50:28] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/home/mingyu/miniconda3/envs/xgboost/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [01:50:28] WARNING: /workspace/src/c_api/c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)


In [21]:
ranker=xgb.Booster()
ranker.load_model('../model/xgboost.model')

In [23]:
test=pl.read_parquet('../data/processData/test.parquet')
test=apply(test,pipeline)
test.head(),test.shape

(shape: (5, 8)
 ┌──────────┬─────────┬────────────┬──────┬──────────────┬──────────────┬─────────────┬─────────────┐
 │ session  ┆ aid     ┆ ts         ┆ type ┆ action_num_r ┆ session_leng ┆ log_recency ┆ type_weight │
 │ ---      ┆ ---     ┆ ---        ┆ ---  ┆ everse_chron ┆ th           ┆ _score      ┆ ed_log_rece │
 │ i32      ┆ i32     ┆ i32        ┆ u8   ┆ o            ┆ ---          ┆ ---         ┆ ncy_scor…   │
 │          ┆         ┆            ┆      ┆ ---          ┆ u32          ┆ f64         ┆ ---         │
 │          ┆         ┆            ┆      ┆ u32          ┆              ┆             ┆ f64         │
 ╞══════════╪═════════╪════════════╪══════╪══════════════╪══════════════╪═════════════╪═════════════╡
 │ 12899779 ┆ 59625   ┆ 1661724000 ┆ 0    ┆ 1            ┆ 1            ┆ 1.0         ┆ 1.0         │
 │ 12899780 ┆ 1142000 ┆ 1661724000 ┆ 0    ┆ 1            ┆ 5            ┆ 0.071773    ┆ 0.071773    │
 │ 12899780 ┆ 582732  ┆ 1661724058 ┆ 0    ┆ 2            ┆ 5       

In [27]:
preds=ranker.predict(xgb.DMatrix(test[feature_cols]))

In [28]:
preds

array([-0.10540748, -0.4211687 ,  0.02247169, ..., -0.28049773,
       -0.21621408, -0.14549842], dtype=float32)

In [32]:
test=test.with_columns(pl.Series(name='score', values=preds)).sort(['session', 'score'], descending=[False, True])
test_predictions = test.group_by('session').agg(
    pl.col('aid').head(20).alias('top20_aid')
)

In [33]:
session_types=[]
labels=[]

for session,preds in zip(test_predictions['session'].to_numpy(),test_predictions['top20_aid'].to_numpy()):
    l=' '.join(str(p) for p in preds)
    for session_type in ['clicks','carts','orders']:
        labels.append(l)
        session_types.append(f'{session}_{session_type}')

In [34]:
submission=pl.DataFrame({'session_type':session_types,'label':labels})
submission.write_csv('../save/xgb_submission.csv')